In [1]:
import json
from pymilvus import MilvusClient
import requests

In [2]:
MILVUS_URI = "http://127.0.0.1:19530"
COLLECTION = "secretarius_semantic_graph"

client = MilvusClient(uri=MILVUS_URI)

if client.has_collection(collection_name=COLLECTION):
    client.drop_collection(collection_name=COLLECTION)
    print(f"Collection supprimée : {COLLECTION}")
else:
    print(f"Collection absente : {COLLECTION}")

Collection supprimée : secretarius_semantic_graph


In [3]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8001/v1", api_key="dummy")

def secretarius(txt:str):
    resp = client.chat.completions.create(
            model="secretarius-notebook",
            messages=[{"role": "user","content":txt}],
        )
    return resp.choices[0].message.content
    
print(secretarius("extraire les expressions caractéristiques de ce texte: Où sont passées les neiges d'antan ?"))

{"expressions": ["les neiges d'antan"], "warning": "chunker returned no chunks; fallback to single raw-text chunk"}


In [4]:
exemples = [
{'texte': "Le paradoxe d'Olbers\nhttps://www.astro-explications.fr/paradoxe-olbers\nPourquoi le ciel nocturne est-il sombre ? Cette question, posée par Heinrich Olbers au XIXe siècle, constitue un paradoxe fascinant. Si l'univers est infini, éternel et peuplé d'une infinité d'étoiles, alors chaque ligne de visée devrait aboutir à la surface d'une étoile. Le ciel entier devrait être aussi brillant que la surface du Soleil, ce qui n'est évidemment pas le cas. La résolution moderne de ce paradoxe repose sur deux piliers : l'univers n'est pas statique mais en expansion, et il a un âge fini. L'expansion étire la lumière des étoiles lointaines vers le rouge, diminuant leur énergie. De plus, la lumière des étoiles les plus éloignées n'a pas encore eu le temps de nous parvenir depuis le Big Bang. Ainsi, l'obscurité de la nuit est une preuve indirecte de la nature dynamique et finie de notre cosmos."},
{'texte': "#écriture #histoire #civilisation\nL'évolution des systèmes d'écriture\n15-03-2023\nL'écriture est née de la nécessité de fixer la pensée et de transmettre l'information au-delà de l'oralité. Les premières traces apparaissent en Mésopotamie avec l'écriture cunéiforme, suivie des hiéroglyphes égyptiens. Chaque civilisation a développé son propre système, des idéogrammes chinois à l'alphabet phénicien, ancêtre de la plupart des écritures modernes. Cette invention a fondamentalement transformé les sociétés, permettant l'administration, la littérature et la préservation du savoir à travers les siècles."},
{'texte': "#temps #physique_quantique #relativité #univers\nLe paradoxe du temps en mécanique quantique\n15-03-2025\nhttps://arxiv.org/abs/quant-ph/2503.08976\nLa notion de temps en physique quantique reste l'une des questions les plus profondes de la science moderne. Alors que la relativité générale d'Einstein décrit le temps comme une dimension flexible, tissée dans le continuum espace-temps, la mécanique quantique le traite souvent comme un paramètre externe et absolu. Cette divergence fondamentale crée des tensions conceptuelles majeures, notamment lorsqu'on tente de formuler une théorie de la gravité quantique.\n\nCertaines approches, comme la gravitation quantique à boucles, proposent une quantification discrète du temps lui-même. D'autres, issues de la théorie des cordes, envisagent le temps comme émergeant d'une réalité sous-jacente plus fondamentale. Des expériences de pensée, comme l'horloge quantique d'Häfele et Keating revisitée avec des états superposés, suggèrent que le flux du temps pourrait être indéfinissable pour un système quantique dans un état de superposition.\n\nLe débat entre un temps illusoire (comme le propose Julian Barbour) et un temps réel et dynamique continue d'animer la communauté des physiciens. La récente expérience utilisant des interféromètres à atomes froids pour tester la dilatation du temps dans un champ gravitationnel faible ouvre peut-être une voie expérimentale pour explorer cette interface mystérieuse entre le quantum et la géométrie de l'espace-temps."},
{'texte': "#Boudienny #cavalerie #guerre_civile #URSS #stratégie\n12-10-1920\nhttps://archives.mil.ru/operations/boudienny_march_1920\n\nLa marche de Boudienny, menée par le commandant de cavalerie soviétique Semion Boudienny, fut une opération décisive durant la guerre civile russe. En octobre 1920, ses unités de cavalerie, les fameux *Konarmia*, percèrent les lignes des Armées Blanches dans le sud de la Russie. Cette manœuvre rapide et audacieuse permit de désorganiser les arrières ennemis et de sécuriser des territoires clés pour l'Armée rouge. L'exploit contribua à renforcer la légende de Boudienny et démontra l'efficacité des grandes formations de cavalerie dans ce conflit."}
]

In [11]:
import json
from IPython.display import Markdown, display

def render_note_graph_markdown(raw_response: str):
    res = json.loads(raw_response)
    query = res.get("query", "")
    docs = res.get("documents", [])

    if not docs:
        display(Markdown(f"## Graphe documentaire pour : `{query}`\n\n_Aucun résultat._"))
        return

    blocks = [f"## Graphe documentaire pour : `{query}`"]

    for i, hit in enumerate(docs, 1):
        doc = hit.get("document", {})
        user = doc.get("user_fields", {})
        source = doc.get("source", {})
        content = doc.get("content", {})
        derived = doc.get("derived", {})

        title = user.get("title") or "(sans titre)"
        type_note = user.get("type_note") or "-"
        date = user.get("document_date") or "-"
        keywords = user.get("keywords", [])
        keyword_matches = hit.get("keyword_matches", [])
        score = hit.get("combined_score", hit.get("score"))
        url = source.get("url")
        text = (content.get("text") or "").strip()
        preview = text[:350] + ("..." if len(text) > 350 else "")

        expressions = derived.get("expressions", [])
        expr_labels = []
        for item in expressions[:5]:
            if isinstance(item, dict):
                expr = item.get("expression")
                if isinstance(expr, str) and expr.strip():
                    expr_labels.append(expr.strip())

        relation_parts = []
        if keyword_matches:
            relation_parts.append("mots-clés communs : " + ", ".join(keyword_matches))
        else:
            relation_parts.append("correspondance sémantique")

        if expr_labels:
            relation_parts.append("expressions saillantes : " + ", ".join(expr_labels))

        block = f"""## {i}. {title}

- Score : `{score}`
- Type : `{type_note}`
- Date : `{date}`
- Mots-clés : {", ".join(keywords) if keywords else "-"}
- Relation : {" ; ".join(relation_parts)}
- URL : {url or "-"}

{preview}
"""
        blocks.append(block)

    display(Markdown("\n\n---\n\n".join(blocks)))


In [6]:
import json
from openai import OpenAI
import time

stime = time.time()
# client API Secretarius
client = OpenAI(
    base_url="http://127.0.0.1:8001/v1",
    api_key="dummy"
)

path = "/home/mauceric/Secretarius/Apprentissage/dspy_gepa/corpus-100.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            # certains de vos fichiers utilisent des quotes Python
            obj = eval(line)

        texte = obj.get("texte")
        if not texte:
            continue

        message = f"/index {texte}"

        resp = client.chat.completions.create(
            model="secretarius-notebook",
            messages=[{"role": "user", "content": message}],
        )
        if i%10 == 0:
            print(f"[{i}] {resp.choices[0].message.content} {time.time() - stime}")

[10] {"status": "ok", "tool": "index_text", "message": "Indexation complete (extraction + insertion semantic graph).", "summary": {"expressions_count": 19, "collection_name": "secretarius_semantic_graph", "inserted_count": 19, "query_count": 19, "hit_lists": 19}, "warning": null} 119.64308333396912
[20] {"status": "ok", "tool": "index_text", "message": "Indexation complete (extraction + insertion semantic graph).", "summary": {"expressions_count": 16, "collection_name": "secretarius_semantic_graph", "inserted_count": 16, "query_count": 16, "hit_lists": 16}, "warning": "filtered out 3 non-verbatim expressions"} 245.24354076385498
[30] {"status": "ok", "tool": "index_text", "message": "Indexation complete (extraction + insertion semantic graph).", "summary": {"expressions_count": 8, "collection_name": "secretarius_semantic_graph", "inserted_count": 8, "query_count": 8, "hit_lists": 8}, "warning": null} 348.63689517974854
[40] {"status": "ok", "tool": "index_text", "message": "Indexation 

In [7]:
for i, exemple in enumerate(exemples):
    print(secretarius(f"/exp {exemple['texte']}"))

{"expressions": ["paradoxe d'Olbers", "ciel nocturne", "Heinrich Olbers", "univers", "étoiles", "Big Bang", "nature dynamique et finie de notre cosmos"]}
{"expressions": ["écriture", "Mésopotamie", "écriture cunéiforme", "hiéroglyphes égyptiens", "civilisation", "idéogrammes chinois", "alphabet phénicien", "sociétés", "administration", "littérature", "préservation du savoir"], "warning": "chunk 0: unable to parse llm json output: Extra data: line 1 column 209 (char 208) | recovered from partial json string list"}
{"expressions": ["temps", "physique quantique", "relativité", "univers", "paradoxe du temps en mécanique quantique", "relativité générale d'Einstein", "mécanique quantique", "théorie des cordes", "réalité sous-jacente", "expériences de pensée", "horloge quantique d'Häfele et Keating", "temps illusoire", "temps réel et dynamique continue", "interféromètres à atomes froids", "dilatation du temps", "champ gravitationnel faible", "interface mystérieuse entre le quantum et la géomé

In [ ]:
for i, exemple in enumerate(exemples):
    print(secretarius(f"/index {exemple['texte']}"))

In [8]:
res = secretarius(f"/req mésopotamie")
res= json.loads(res)
#print(res)
for i, document in enumerate(res['documents']) :
    print(document['document']['content'])

{'text': "L'écriture est née il y a plus de cinq mille ans en Mésopotamie, sous la forme de pictogrammes cunéiformes sur des tablettes d'argile. Cette invention majeure a révolutionné la transmission des connaissances, permettant de fixer la langue et de développer l'administration, le commerce et la littérature. Différents systèmes ont émergé indépendamment à travers le monde, comme les hiéroglyphes égyptiens, les caractères chinois ou les glyphes mayas. Chaque système reflète la culture et les besoins de la société qui l'a créé. L'évolution vers des alphabets phonétiques, comme l'alphabet phénicien puis grec, a simplifié l'apprentissage et favorisé la diffusion de l'écrit. Aujourd'hui, l'écriture numérique poursuit cette longue histoire, transformant à nouveau nos modes de communication et de conservation de la mémoire collective.", 'hash': 'sha256:339efc948ebcd4b2dbb38dc831e89fd5d561c0b3f3c2e23dbbf72767a6a20e86'}
{'text': "L'écriture est née de la nécessité de fixer la pensée et de 

In [9]:
res = secretarius(f"/req cavalerie")
res= json.loads(res)
#print(res)
for i, document in enumerate(res['documents']) :
    print(document['document']['user_fields'])

{'type_note': 'fugace', 'title': 'La marche de Boudienny, menée par le...', 'document_date': '12-10-1920', 'keywords': ['#Boudienny', '#cavalerie', '#guerre_civile', '#URSS', '#stratégie'], 'updated_at': '2026-03-14T16:43:56Z', 'created_at': '2026-03-14T16:43:46Z'}
{'type_note': 'fugace', 'title': 'La Marche de Boudienny', 'document_date': '12-10-1920', 'keywords': [], 'updated_at': '2026-03-14T16:40:02Z', 'created_at': '2026-03-14T16:39:55Z'}


In [14]:
res = secretarius("/req cavalerie")
#render_note_graph_markdown(res)
print(res)

{"status": "ok", "tool": "search_text", "message": "Recherche semantique executee.", "query": "cavalerie", "summary": {"collection_name": "secretarius_semantic_graph", "query_count": 1, "hit_lists": 1, "top_k": 10, "min_score": 0.75, "keyword_query_count": 0}, "documents": [{"id": 8048787132151029227, "score": 0.962354838848114, "combined_score": 0.962354838848114, "keyword_matches": [], "document": {"schema": "secretarius.document.v0.1", "doc_id": "doc:3da739c3da977f3131bb88db64c4eb2f3ffc4fc79c50d4e1d074af8cb71dde7d", "type": "url", "source": {"urls": ["https://archives.mil.ru/operations/boudienny_march_1920"], "url": "https://archives.mil.ru/operations/boudienny_march_1920", "source_id": "src:af6d2d2f9bc47e6e3c3066ad933735eec7e6530db328d38399d41062f589be6d"}, "content": {"text": "La marche de Boudienny, menée par le commandant de cavalerie soviétique Semion Boudienny, fut une opération décisive durant la guerre civile russe. En octobre 1920, ses unités de cavalerie, les fameux *Konar

In [13]:
print(secretarius("/req cavalerie"))

{"status": "ok", "tool": "search_text", "message": "Recherche semantique executee.", "query": "cavalerie", "summary": {"collection_name": "secretarius_semantic_graph", "query_count": 1, "hit_lists": 1, "top_k": 10, "min_score": 0.75, "keyword_query_count": 0}, "documents": [{"id": 8048787132151029227, "score": 0.962354838848114, "combined_score": 0.962354838848114, "keyword_matches": [], "document": {"schema": "secretarius.document.v0.1", "doc_id": "doc:3da739c3da977f3131bb88db64c4eb2f3ffc4fc79c50d4e1d074af8cb71dde7d", "type": "url", "source": {"urls": ["https://archives.mil.ru/operations/boudienny_march_1920"], "url": "https://archives.mil.ru/operations/boudienny_march_1920", "source_id": "src:af6d2d2f9bc47e6e3c3066ad933735eec7e6530db328d38399d41062f589be6d"}, "content": {"text": "La marche de Boudienny, menée par le commandant de cavalerie soviétique Semion Boudienny, fut une opération décisive durant la guerre civile russe. En octobre 1920, ses unités de cavalerie, les fameux *Konar

In [ ]:
import 